In [ ]:
import optuna

storage_path = "sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db"
summaries = optuna.get_all_study_summaries(storage=storage_path)
for s in summaries:
    print(f"study_name: {s.study_name}, n_trials: {s.n_trials}")

study_name: latentgee_optuna_df1, n_trials: 2000
study_name: latentgee_optuna_df3, n_trials: 2000
study_name: latentgee_optuna_df4, n_trials: 2000


In [2]:
import optuna

for db_path in [
    "sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db",
    "sqlite:////DATA/WGS_study/YSL/projects/latentgee/latentgee_optuna.db"
]:
    print(f"\n=== {db_path} ===")
    summaries = optuna.get_all_study_summaries(storage=db_path)
    for s in summaries:
        print(f"study_name: {s.study_name}, n_trials: {s.n_trials}")


=== sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db ===
study_name: latentgee_optuna_df1, n_trials: 2000
study_name: latentgee_optuna_df3, n_trials: 2000
study_name: latentgee_optuna_df4, n_trials: 2000

=== sqlite:////DATA/WGS_study/YSL/projects/latentgee/latentgee_optuna.db ===


In [3]:
import optuna
import pandas as pd

study = optuna.load_study(
    study_name="latentgee_optuna_df1",
    storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/latentgee_optuna.db"
)

# trial 날짜 분포 확인
dates = [t.datetime_start.date() for t in study.trials if t.datetime_start]
date_counts = pd.Series(dates).value_counts().sort_index()
print(date_counts)

KeyError: 'Record does not exist.'

In [7]:
import optuna
import pandas as pd

study = optuna.load_study(
    study_name="latentgee_optuna_df1",
    storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/latentgee_optuna.db"
)

# 5월 12일 이후 trial만 필터링
from datetime import date

valid_trials = [
    t for t in study.trials
    if t.datetime_start
    and t.datetime_start.date() >= date(2026, 5, 12)
    and t.state == optuna.trial.TrialState.COMPLETE
    and t.value is not None
]

# DataFrame으로 변환
rows = []
for t in valid_trials:
    row = {"trial_number": t.number, "permanova": t.value}
    row.update(t.params)
    rows.append(row)

df_valid = pd.DataFrame(rows)
print(f"유효 trial 수: {len(df_valid)}")

# best trial 필터링 (이전 기준)
df_clean = df_valid[
    (df_valid['permanova'] > 0.01) &
    (df_valid['permanova'] < 0.1) &
    (df_valid['noise_ratio'] < 0.65) &
    (df_valid['n_clusters'] >= 10)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'],
              ascending=[True, True, False]).head(20)

print(f"필터링 후 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

유효 trial 수: 5784


KeyError: 'noise_ratio'

In [8]:
print(df_valid.columns.tolist())
print(df_valid.head(2))

['trial_number', 'permanova', 'cutoff', 'init', 'beta_kl', 'norm', 'strategy', 'n_layers', 'base_dim', 'latent_dim', 'activation', 'dropout_rate', 'epochs', 'batch_size', 'weight_decay', 'learning_rate', 'grad_clip_norm', 'kl_warmup_ratio', 'cluster_selection_method', 'min_cluster_size', 'min_samples_token', 'scheduler']
   trial_number  permanova  cutoff           init   beta_kl       norm  \
0           645   0.107612    0.05  xavier_normal  0.012176  layernorm   
1           646   0.053172    0.05  xavier_normal  0.016199       none   

   strategy  n_layers  base_dim  latent_dim  ... epochs  batch_size  \
0  constant         2       768          32  ...    100          64   
1  constant         2       768          32  ...    100          64   

   weight_decay  learning_rate  grad_clip_norm  kl_warmup_ratio  \
0  1.083572e-07       0.000495        0.563364         0.654953   
1  1.763496e-07       0.000531        0.563660         0.662449   

   cluster_selection_method  min_clust